# TP2 - Traitement des données avec Pandas (Google Colab)

Notebook réalisé à partir du sujet **TP 2 - Traitement des données avec Pandas**.

**Objectifs couverts :**
- chargement et inspection,
- nettoyage des valeurs manquantes,
- indexation et filtres,
- création d'indicateurs,
- agrégations (`groupby`),
- jointure (`merge`),
- tableau croisé (`pivot_table`),
- visualisations et conclusion métier.

In [ ]:
# Colab: installer les dépendances si besoin
%pip -q install pandas matplotlib openpyxl

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## Création des fichiers CSV demandés dans le sujet
Le notebook génère `ventes.csv` et `regions.csv` pour être autonome sur Colab.

In [ ]:
ventes_csv = """Date,Produit,Categorie,Region,Quantite,Prix_Unitaire
2024-01-05,PC Portable,Informatique,Casablanca,4,8500
2024-01-07,Souris,Accessoire,Rabat,20,120
2024-01-12,Clavier,Accessoire,Casablanca,15,250
2024-01-15,Ecran,Informatique,Tanger,6,1800
2024-01-18,PC Portable,Informatique,Rabat,,8500
2024-01-20,Imprimante,Informatique,Casablanca,3,2200
2024-01-25,Souris,Accessoire,Tanger,30,120
2024-01-28,Clavier,Accessoire,Rabat,10,
2024-02-02,Ecran,Informatique,Casablanca,5,1800
2024-02-05,Imprimante,Informatique,Tanger,2,2200
"""

regions_csv = """Region,Responsable,Objectif_CA
Casablanca,Amina,50000
Rabat,Yassine,35000
Tanger,Hajar,25000
"""

Path("ventes.csv").write_text(ventes_csv, encoding="utf-8")
Path("regions.csv").write_text(regions_csv, encoding="utf-8")
print("Fichiers créés.")

## Exercice 1 - Chargement et inspection

In [ ]:
df = pd.read_csv("ventes.csv")
regions = pd.read_csv("regions.csv")

print("Aperçu ventes:")
print(df.head())
print()

print("Info DataFrame ventes:")
print(df.info())
print()

print("Valeurs manquantes par colonne:")
print(df.isnull().sum())
print("Total valeurs manquantes:", int(df.isnull().sum().sum()))

### Réponses Exercice 1
- Colonnes avec valeurs manquantes : **Quantite** et **Prix_Unitaire**.
- Nombre total de valeurs manquantes : **2**.
- Les valeurs manquantes peuvent biaiser des moyennes, sommes et comparaisons.
- Différences :
  - `info()` : structure, types, non-nuls, mémoire.
  - `describe()` : statistiques descriptives.
  - `isnull().sum()` : comptage précis des valeurs manquantes par colonne.

## Exercice 2 - Nettoyage des valeurs manquantes

In [ ]:
# Remplacer Quantite par la moyenne
df["Quantite"] = df["Quantite"].fillna(df["Quantite"].mean())

# Remplacer Prix_Unitaire par la médiane
df["Prix_Unitaire"] = df["Prix_Unitaire"].fillna(df["Prix_Unitaire"].median())

print("Valeurs manquantes après nettoyage:")
print(df.isnull().sum())

### Réponses Exercice 2
- La médiane est préférable en présence de valeurs extrêmes car elle est plus robuste.
- Cette stratégie n'est pas toujours idéale : elle peut masquer des causes métier des absences.
- En contexte réel, on peut aussi utiliser l'imputation par groupe (produit/région), la suppression contrôlée, ou une règle métier validée.

## Exercice 3 - Indexation et interrogation

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df_date = df.set_index("Date")

print("Ventes de janvier 2024:")
print(df_date.loc["2024-01"])
print()

print("Sélection colonnes Produit/Region/Quantite:")
print(df[["Produit", "Region", "Quantite"]])
print()

print("Filtre Region == Casablanca:")
print(df[df["Region"] == "Casablanca"])
print()

filtre = (df["Region"] == "Casablanca") & (df["Categorie"] == "Informatique")
print("Filtre combiné Casablanca + Informatique:")
print(df[filtre])
print()

nb_ventes_info_casa = int(filtre.sum())
print("Nombre de ventes Informatique à Casablanca:", nb_ventes_info_casa)

### Réponses Exercice 3
- `datetime` permet des filtres temporels puissants (par mois, période, tri chronologique).
- `loc` indexe par étiquette (ici date), alors qu'un filtre conditionnel sélectionne via une expression booléenne sur colonnes.
- Ventes Informatique à Casablanca : **3**.

## Exercice 4 - Création d'indicateurs

In [ ]:
df["Chiffre_Affaires"] = df["Quantite"] * df["Prix_Unitaire"]
df["Mois"] = df["Date"].dt.to_period("M")

print(df[["Date", "Produit", "Region", "Chiffre_Affaires", "Mois"]])

idx_max_ligne = df["Chiffre_Affaires"].idxmax()
produit_max_ligne = df.loc[idx_max_ligne, "Produit"]
ca_total = df["Chiffre_Affaires"].sum()

print("Produit avec CA ligne max:", produit_max_ligne)
print("CA total de l'entreprise:", ca_total)

### Réponses Exercice 4
- Produit générant le CA le plus élevé sur une ligne : **PC Portable**.
- Chiffre d'affaires total : calculé dans la cellule ci-dessus.
- La variable `Mois` facilite l'analyse temporelle et les comparaisons mensuelles.

## Exercice 5 - Groupby, tri et visualisations

In [ ]:
ca_region = df.groupby("Region")["Chiffre_Affaires"].sum().sort_values(ascending=False)
ca_produit = df.groupby("Produit")["Chiffre_Affaires"].sum().sort_values(ascending=False)
ca_categorie = df.groupby("Categorie")["Chiffre_Affaires"].sum().sort_values(ascending=False)

print("CA par région:")
print(ca_region)
print()

print("CA par produit:")
print(ca_produit)
print()

print("CA par catégorie:")
print(ca_categorie)

In [ ]:
# Graphique 1 : CA par région
ca_region.plot(kind="bar", figsize=(7, 4), color="#4C78A8")
plt.title("Chiffre d'affaires par région")
plt.ylabel("Chiffre d'affaires")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Graphique 2 : CA par produit
ca_produit.plot(kind="bar", figsize=(8, 4), color="#F58518")
plt.title("Chiffre d'affaires par produit")
plt.ylabel("Chiffre d'affaires")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Réponses Exercice 5
- Région la plus performante : la première de `ca_region`.
- Produit le plus rentable : le premier de `ca_produit`.
- La catégorie Informatique domine selon les montants agrégés affichés.

## Exercice 6 - Jointure avec `merge`

In [ ]:
df_complet = pd.merge(df, regions, on="Region", how="left")
print("Aperçu dataframe joint:")
print(df_complet.head())
print()

ca_objectif = (
    df_complet.groupby(["Region", "Responsable", "Objectif_CA"])["Chiffre_Affaires"]
    .sum()
    .reset_index()
)

ca_objectif["Taux_Realisation"] = ca_objectif["Chiffre_Affaires"] / ca_objectif["Objectif_CA"] * 100
print("CA vs Objectif:")
print(ca_objectif)

### Réponses Exercice 6
- Clé de jointure : **Region**.
- `how="left"` conserve toutes les lignes de `df` (table de gauche).
- Meilleur taux de réalisation : la région avec le `Taux_Realisation` maximal.
- Recommandation pour la région la moins performante : renforcer les actions commerciales locales et se concentrer sur les produits à forte marge.

## Exercice 7 - Tableau croisé dynamique

In [ ]:
pivot = pd.pivot_table(
    df,
    values="Chiffre_Affaires",
    index="Region",
    columns="Categorie",
    aggfunc="sum",
    fill_value=0,
)

print("Pivot CA par Region x Categorie:")
print(pivot)

region_plus_accessoire = pivot["Accessoire"].idxmax() if "Accessoire" in pivot.columns else None
cat_dominante_rabat = pivot.loc["Rabat"].idxmax() if "Rabat" in pivot.index else None

print("Région vendant le plus d'Accessoire:", region_plus_accessoire)
print("Catégorie dominante à Rabat:", cat_dominante_rabat)

### Réponses Exercice 7
- La région qui vend le plus d'Accessoire est affichée dans la cellule précédente.
- La catégorie dominante à Rabat est également affichée ci-dessus.
- Le pivot rend la comparaison croisée **région × catégorie** immédiate, plus lisible qu'un tableau brut ligne à ligne.

## Conclusion métier (10 lignes)
1. Les données brutes contenaient des valeurs manquantes sur la quantité et le prix unitaire.
2. Le nettoyage par moyenne/médiane a permis de conserver toutes les ventes pour l'analyse.
3. Le chiffre d'affaires total donne une vue consolidée de la performance globale.
4. L'analyse par région met en évidence une zone la plus performante en contribution au CA.
5. L'analyse par produit montre un produit leader en rentabilité.
6. La catégorie Informatique représente la part principale du chiffre d'affaires.
7. La jointure avec les objectifs régionaux permet de mesurer le taux de réalisation commercial.
8. Une région ressort avec un retard relatif sur son objectif et nécessite un plan d'action.
9. Les recommandations portent sur le mix produit, la priorisation commerciale et le suivi mensuel.
10. La qualité de la saisie doit être améliorée pour limiter les imputations futures et fiabiliser le pilotage.